# Trendyol-LLM koc olcumu

`Trendyol/Trendyol-LLM-7B-chat-v4.1.0` modeline NEXORA API'sinin **gercek** koc
prompt'unu atar, sema tutturma oranini ve gecikmeyi olcer.

**Bu bir olcum kosumu, demo altyapisi degil.** Juri demosunun varsayilani canned
Turkce fallback olarak kalir (`docs/DEMO.md`).

## Neden

`docs/ARCHITECTURE.md` "LLM boundary" ve `docs/building-blocks/04-llm-coach.md`
olctu: `router.huggingface.co` 143 model sunuyor, hicbiri Trendyol degil. Urun
dokumani Trendyol-LLM vaat ediyor, mimari dokumani "saglanmiyor" diyor, arada
veri yok. Bu defter o veriyi uretir.

## Neden vLLM yok

Ilk surum vLLM sunucusu + cloudflared tuneli kuruyordu. Colab'da uc yerden
kiriliyor: `pip install vllm` Colab'in torch'unu degistirip runtime restart
gerektiriyor, T4'un compute capability'si 7.5 ama yeni vLLM motoru >= 8.0
istiyor, ve bitsandbytes ayrica kurulmali. Olcum icin sunucuya gerek yok -
uc prompt'un cevabini istiyoruz. Model dogrudan `transformers` ile yuklenir;
uctan uca denemek isteyen icin ayni yuklu modeli sunan kucuk bir shim var.

## Repo'dan buraya ne geliyor

Sadece iki metin: `apps/api/src/coach/index.ts`'teki `SYSTEM_PROMPT` ve
`buildUserPrompt` ciktisi. Ikisi de asagida birebir kopyali. Veritabani yok,
gercek kullanici yok, URL yok.

## Sonra ne olacak

Son hucrenin bastigi tabloyu `docs/building-blocks/04-llm-coach.md`'ye,
`Qwen/Qwen3-8B` olcumunun yanina tarihli bir not olarak yaz. Asil teslimat o.

> **Runtime > Change runtime type > T4 GPU.** Commit'ten once
> `Runtime > Restart and clear all outputs`.

## 1 - Ortam

`torch` ve `transformers` Colab'da zaten kurulu; sadece 4-bit icin eksik olan
ikisi geliyor. Torch'a dokunulmadigi icin runtime restart gerekmez.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q bitsandbytes accelerate

## 2 - Model

7B fp16 ~15 GB; T4'un 15 GB'ina sigmaz, o yuzden 4-bit NF4. Compute dtype
`float16`: T4 Turing mimarisi, bf16 yok. Agirlik indirmesi ~15 GB, ilk kosuda
5-10 dk surer.

Model herkese aciktir, token gerekmez. "You are sending unauthenticated requests"
uyarisi bir hata degil - indirme yine yurur, sadece yavas ve rate limit'e tabi.
Hizlandirmak icin token eklemek istersen Colab'in sol kenarindaki Secrets
panelinde `HF_TOKEN` olustur (bu defter icin erisimi ac) ve bu hucreden once
calistir - token'i asla duz metin yazma:

```python
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))
```

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Trendyol/Trendyol-LLM-7B-chat-v4.1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
    ),
)
model.eval()
print(model.device, next(model.parameters()).dtype)
print(f"{torch.cuda.memory_allocated() / 1e9:.1f} GB allocated")

## 3 - Prompt'lar

`SYSTEM_PROMPT` birebir `apps/api/src/coach/index.ts`'ten. Uc kullanici prompt'u
`buildUserPrompt`'un gercek ciktisi - `packages/score`'un uc fixture'i
(`docs/building-blocks/03-signals-score.md`: risky 38, balanced 80, productive 93)
`computeScore`'dan gecirilip kopyalandi.

In [ ]:
SYSTEM_PROMPT = "\n".join([
    "Sen 13-18 yaş arası gençler için Türkçe yazan bir dijital denge koçusun.",
    "Teşhis koymazsın, etiketlemezsin, suçlamazsın; kısa ve uygulanabilir öneriler verirsin.",
    "Yalnızca tek bir JSON nesnesi döndür, başka hiçbir metin yazma:",
    '{"tips":["","",""],"task":{"title":"","steps":["",""],"eta_minutes":15},"share_text":""}',
    "Kurallar: tips tam 3 madde; steps 2-5 madde; eta_minutes 5-30 arası tam sayı;",
    "hiçbir metinde bağlantı adresi olmasın; share_text veliyle paylaşılabilecek tek cümle olsun.",
])

# buildUserPrompt() output for the three score fixtures, verbatim.
PROMPTS = {
    "risky (38)": '{"minutes":{"science":10,"entertainment":200,"harmful":40},"score":{"value":38,"reasons":["Zararlı veya manipülatif kategoride süre var; bunu bir yetişkinle konuşmak iyi olabilir.","Eğlence kategorisi sürenin çoğunu kaplıyor.","Kategori çeşitliliğin düşük; kısa bir keşif görevi dene."]},"completed_tasks":[]}',
    "balanced (80)": '{"minutes":{"science":40,"arts":15,"sports":20,"culture":10,"national_memory":5,"entertainment":120},"score":{"value":80,"reasons":["Eğlence kategorisi sürenin çoğunu kaplıyor.","Bilim, sanat, spor veya kültür kategorilerinde görünür bir payın var.","Birden fazla değerli kategoride zaman geçirmişsin."]},"completed_tasks":[]}',
    "productive (93)": '{"minutes":{"science":70,"arts":20,"sports":20,"culture":15,"entertainment":40},"score":{"value":93,"reasons":["Bilim, sanat, spor veya kültür kategorilerinde görünür bir payın var.","Birden fazla değerli kategoride zaman geçirmişsin.","Zararlı/manipülatif kategoride süre görünmüyor."]},"completed_tasks":[]}',
}

# packages/shared coachSchema, as plain asserts. z.strictObject => no extra keys.
def schema_error(c):
    if not isinstance(c, dict) or set(c) != {"tips", "task", "share_text"}:
        return "top-level keys"
    if not (isinstance(c["tips"], list) and len(c["tips"]) == 3):
        return "tips != 3"
    if any(not isinstance(t, str) or not t for t in c["tips"]):
        return "empty tip"
    t = c["task"]
    if not isinstance(t, dict) or set(t) != {"title", "steps", "eta_minutes"}:
        return "task keys"
    if not isinstance(t["title"], str) or not t["title"]:
        return "task.title"
    if not (isinstance(t["steps"], list) and 2 <= len(t["steps"]) <= 5):
        return "steps out of 2-5"
    if any(not isinstance(s, str) or not s for s in t["steps"]):
        return "empty step"
    if not isinstance(t["eta_minutes"], int) or isinstance(t["eta_minutes"], bool):
        return "eta not int"
    if not 5 <= t["eta_minutes"] <= 30:
        return "eta out of 5-30"
    if not isinstance(c["share_text"], str) or not c["share_text"]:
        return "share_text"
    return None

# coach/index.ts BANNED, verbatim.
BANNED = ["teşhis", "teshis", "depresyon tanısı", "depresyon tanisi", "bağımlısın",
          "bagimlisin", "kötü çocuk", "kotu cocuk", "http://", "https://"]

def generate(user_prompt):
    """Same knobs as coach/index.ts modelCoach(): temperature 0.3, 900 new tokens."""
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=900,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

## 4 - Bir deneme

Otuz cagriyi beklemeden once tek bir cevabin makul gorundugunu dogrula. Cikti
sacmaliyorsa sorun promptta degil chat template'te olabilir.

In [ ]:
print(generate(PROMPTS["balanced (80)"]))

## 5 - Olcum

Her bant icin N cagri. Gecikme `transformers` uzerinden olculur; vLLM gibi bir
sunucu bunu bir miktar kisaltirdi, yani buradaki sayilar **ust sinir**. Sema
tutturma orani icin boyle bir fark yok, cevap dagilimi ayni.

In [ ]:
import json, statistics, time

N = 10  # calls per band; 30 total. Raise once you know one call's cost.

results = {}
for band, prompt in PROMPTS.items():
    lat, ok, fenced, banned, errs = [], 0, 0, 0, []
    for i in range(N):
        began = time.time()
        try:
            text = generate(prompt)
        except Exception as e:  # OOM, template error, anything
            errs.append(type(e).__name__)
            continue
        lat.append(time.time() - began)
        # coach/index.ts: models wrap JSON in prose or ``` fences.
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1:
            errs.append("no JSON object")
            continue
        if start > 0 or end < len(text.rstrip()) - 1:
            fenced += 1
        try:
            coach = json.loads(text[start:end + 1])
        except ValueError:
            errs.append("not JSON")
            continue
        why = schema_error(coach)
        if why:
            errs.append(why)
            continue
        if any(b in json.dumps(coach, ensure_ascii=False).lower() for b in BANNED):
            banned += 1
            errs.append("banned phrase")
            continue
        ok += 1
    results[band] = {
        "n": N, "ok": ok, "fenced": fenced, "banned": banned,
        "p50": statistics.median(lat) if lat else None,
        "p95": sorted(lat)[max(0, int(len(lat) * 0.95) - 1)] if lat else None,
        "errors": errs,
    }
    print(band, results[band])

In [ ]:
from collections import Counter

print(f"{MODEL}  |  T4, 4-bit nf4, fp16 compute  |  transformers  |  N={N} per band\n")
print(f"{'band':<18}{'schema ok':>11}{'p50 s':>9}{'p95 s':>9}{'fenced':>9}{'banned':>9}")
for band, r in results.items():
    p50 = f"{r['p50']:.1f}" if r["p50"] else "-"
    p95 = f"{r['p95']:.1f}" if r["p95"] else "-"
    print(f"{band:<18}{str(r['ok']) + '/' + str(r['n']):>11}{p50:>9}{p95:>9}{r['fenced']:>9}{r['banned']:>9}")

total_ok = sum(r["ok"] for r in results.values())
total_n = sum(r["n"] for r in results.values())
print(f"\noverall schema hit rate: {total_ok}/{total_n} = {total_ok / total_n:.0%}")
print("Qwen/Qwen3-8B via the HF router measured ~3/4 (docs/building-blocks/04-llm-coach.md).")
print("The API gives up at 20s (AbortSignal.timeout) and falls back.")
print("\nfailures:", Counter(e for r in results.values() for e in r["errors"]).most_common())

## 6 - Uctan uca (opsiyonel)

Sayilari aldiysan is bitti. API'yi gercekten bu modele baglamak istersen asagisi
zaten yuklu modeli OpenAI-sekilli tek bir uc noktadan sunar. Ikinci bir indirme
yok, vLLM yok.

Shim sadece `coach/index.ts`'in okudugu alani doner: `choices[0].message.content`.

In [ ]:
import json, re, subprocess, threading, time
from http.server import BaseHTTPRequestHandler, HTTPServer

PORT = 8000
API_KEY = "nexora-local-key"

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.headers.get("authorization") != f"Bearer {API_KEY}":
            self.send_response(401); self.end_headers(); return
        if not self.path.endswith("/chat/completions"):
            self.send_response(404); self.end_headers(); return
        body = json.loads(self.rfile.read(int(self.headers["content-length"])))
        # The shim honours the messages only; the sampling knobs are fixed above.
        user = next(m["content"] for m in reversed(body["messages"]) if m["role"] == "user")
        content = generate(user)
        out = json.dumps({"choices": [{"message": {"content": content}}]}).encode()
        self.send_response(200)
        self.send_header("content-type", "application/json")
        self.send_header("content-length", str(len(out)))
        self.end_headers()
        self.wfile.write(out)

    def log_message(self, *a):  # keep the notebook readable
        pass

threading.Thread(target=HTTPServer(("0.0.0.0", PORT), Handler).serve_forever, daemon=True).start()

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT,
)

public, deadline = None, time.time() + 60
while time.time() < deadline and not public:
    time.sleep(2)
    hit = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    public = hit.group(0) if hit else None
if not public:
    print(open("cloudflared.log").read()[-2000:])
    raise SystemExit("no tunnel url")

print("Paste into .env at the repo root, then `bun run dev --filter nexora-api`:\n")
print(f"HF_BASE_URL={public}/v1")
print(f"HF_TOKEN={API_KEY}")
print(f"HF_MODEL_ID={MODEL}")

Dogrulama (`docs/DEMO.md` personasi `deniz`, aktif ve skoru olan):

```
GET /coach/recommendation   ->   200, X-Nexora-Coach: model, source: "model"
```

Sonra bu defterin calisma zamanini **durdur** ve ayni cagriyi tekrarla:

```
GET /coach/recommendation   ->   200, X-Nexora-Coach: fallback
```

Ikincisi asil testtir: yeni upstream dustugunde de fail-closed davraniyor mu.

Bittiginde `.env`'den uc satiri sil - bayat bir tunel URL'si her koc cagrisina
20 saniye ekler.